In [20]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import re
import time
from datetime import datetime, timedelta
import os

# Initialize storage for article data and Parquet file path
article_data = []
parquet_file = "../../data/00-newspaper_data/crawler/altavoz/articles.parquet"

# Base URL format for daily pages
base_url = "https://altavz.com/{year}/{month:02d}/{day:02d}/"

def is_relevant_url(url):
    """
    Check if the URL matches the pattern YYYY/MM/DD/title.
    """
    pattern = r'\d{4}/\d{2}/\d{2}/[a-zA-Z0-9-]+'
    return re.search(pattern, url)

def extract_article_data(url):
    """
    Extract and return the title, main text, date, and source from an article page.
    """
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        # Extract the title
        title_tag = soup.find('h1', class_='entry-title')
        title = title_tag.get_text(strip=True) if title_tag else 'No title found'

        # Extract the main text
        main_content = soup.find_all('p')
        main_text = ' '.join(p.get_text(strip=True) for p in main_content)

        # Extract the date
        date_tag = soup.find('time')
        date = date_tag['datetime'] if date_tag and 'datetime' in date_tag.attrs else 'No date found'

        # Extract the source
        source_tag = soup.find('em')
        source = ''
        if source_tag:
            strong_tag = source_tag.find('strong')
            source = strong_tag.get_text(strip=True) if strong_tag else 'No source found'

        return {
            'url': url,
            'title': title,
            'main_text': main_text,
            'date': date,
            'source': source
        }
    
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data from {url}: {e}")
        return None

def crawl_daily_page(year, month, day):
    """
    Crawl a daily page and find all article links.
    """
    url = base_url.format(year=year, month=month, day=day)
    daily_data = []  # Store daily data temporarily for this day
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        

        print(f"Accessing daily page: {url}")

        # Find and process all article links
        # Deduplicate links by collecting them in a set
        unique_links = set()
        for link in soup.find_all("a", href=True):
            href = urljoin(url, link['href'])
            if is_relevant_url(href):
                unique_links.add(href)  # Add only unique links

        # Extract and store article data from unique pages
        for href in unique_links:
            article = extract_article_data(href)
            if article:
                daily_data.append(article)
                print(f"Extracted data from {href}")

        # Wait to avoid overloading server
        time.sleep(2)

    except requests.exceptions.RequestException as e:
        print(f"Error accessing daily page {url}: {e}")

    return daily_data

# Function to save data to parquet
def save_to_parquet(data, parquet_file):
    df = pd.DataFrame(data)
    if not df.empty:
        if os.path.exists(parquet_file):
            initial = pd.read_parquet(parquet_file)
            df = pd.concat([initial, df]).reset_index(drop=True)
            df.to_parquet(parquet_file, engine="pyarrow", compression="gzip")
        else:
            df.to_parquet(parquet_file, engine="pyarrow", compression="gzip")

# Determine the start date by checking the existing Parquet file
if os.path.exists(parquet_file):
    existing_data = pd.read_parquet(parquet_file)
    last_date_str = existing_data['date'].max()
    last_date = datetime.strptime(last_date_str, "%Y-%m-%d")
    start_date = last_date + timedelta(days=1)
    print(f"Resuming crawl from {start_date.date()}")
else:
    start_date = datetime(2012, 1, 1)

# Set the end date to January 31, 2024
end_date = datetime(2024, 10, 30)

# Crawl and save articles day-by-day, resuming if interrupted
current_date = start_date
while current_date <= end_date:
    daily_data = crawl_daily_page(current_date.year, current_date.month, current_date.day)
    if daily_data:
        save_to_parquet(daily_data, parquet_file)
    current_date += timedelta(days=1)
    print(f"Completed crawling for {current_date.date() - timedelta(days=1)}")

print("Crawling completed or paused; data saved in Parquet format.")





Error accessing daily page https://altavz.com/2012/01/01/: 404 Client Error: Not Found for url: https://altavz.com/2012/01/01/
Completed crawling for 2012-01-01
Error accessing daily page https://altavz.com/2012/01/02/: 404 Client Error: Not Found for url: https://altavz.com/2012/01/02/
Completed crawling for 2012-01-02
Error accessing daily page https://altavz.com/2012/01/03/: 404 Client Error: Not Found for url: https://altavz.com/2012/01/03/
Completed crawling for 2012-01-03
Error accessing daily page https://altavz.com/2012/01/04/: 404 Client Error: Not Found for url: https://altavz.com/2012/01/04/
Completed crawling for 2012-01-04
Error accessing daily page https://altavz.com/2012/01/05/: 404 Client Error: Not Found for url: https://altavz.com/2012/01/05/
Completed crawling for 2012-01-05
Error accessing daily page https://altavz.com/2012/01/06/: 404 Client Error: Not Found for url: https://altavz.com/2012/01/06/
Completed crawling for 2012-01-06
Error accessing daily page https:/

In [21]:
ddd = pd.read_parquet(parquet_file)
#ddd['main_text'][0]
ddd

,url,title,main_text,date,source
0,https://altavz.com/2015/10/05/reflexion-sobre-...,Reflexión sobre la Tenencia,I. Es importante recordar el origen de la tene...,2015-10-05T00:00:33-05:00,
1,https://altavz.com/2015/10/05/romper-el-paradi...,ROMPER EL PARADIGMA DE MOVILIDAD ¿DE QUIÉN SON...,Uno de los temas sobre el cual todos siempre ...,2015-10-05T00:00:05-05:00,
2,https://altavz.com/2015/10/05/los-medina-se-fu...,¿Los Medina se Fugan?,Ahora el ex gobernador Medina y su señora anun...,2015-10-05T13:00:33-05:00,
3,https://altavz.com/2015/10/05/hayotromexicoenc...,#HayOtroMéxicoEnCurso,"La Escuela de Gobierno del Tec de Monterrey, l...",2015-10-05T00:00:13-05:00,
4,https://altavz.com/2015/10/05/a-jalar-todos-qu...,"¡A jalar todos, que se ocupa!",El pasado sábado el Ing. Jaime Rodríguez Calde...,2015-10-05T00:00:51-05:00,
...,...,...,...,...,...
9891,https://altavz.com/2024/10/18/comision-de-segu...,Comisión de Seguimiento a la Implementación de...,"El 15 de octubre, con gran responsabilidad ant...",2024-10-18T08:52:48-06:00,
9892,https://altavz.com/2024/10/21/la-seguridad-no-...,LA SEGURIDAD NO SE POLITIZA,La seguridad en México y en cada uno de los Es...,2024-10-21T11:01:54-06:00,
9893,https://altavz.com/2024/10/24/dos-anos-perdido...,Dos años perdidos y un nuevo acuerdo en favor ...,Después de 2 años de nula relación entre el Po...,2024-10-24T08:10:56-06:00,
9894,https://altavz.com/2024/10/28/harris-vs-trump-...,Harris vs Trump: Una batalla por el alma de Am...,Faltan solo un par de días para que EE. UU. vu...,2024-10-28T08:34:37-06:00,
